# Disjoint Set Union (DSU) – Union Find Algorithm

## 1. What is DSU?

- DSU stands for **Disjoint Set Union**.
- Also known as the **Union-Find Algorithm**.
- It is used to manage multiple disjoint (non-overlapping) sets.
- Initially, all elements belong to separate sets.
- We keep combining sets and deciding their leaders (representatives).

### Two Main Operations

DSU supports only two main operations:

1. **Union(a, b):** Combine the sets containing `a` and `b`.
2. **Find(a):** Find the representative (leader/parent) of the set containing `a`.

To check whether two elements belong to the same set:

```cpp
find(a, parent) == find(b, parent)
```

- If equal → Same set.
- If not equal → Different sets.

---

## 2. Initialisation

We use two vectors:

1. `Element` – Represents the elements (usually their indices).
2. `Parent` – Stores the parent of every element.

Initially:

- Every element belongs to a separate set.
- Every node is its own parent.
- Every element is its own leader.

```cpp
vector<int> parent(n);

for(int i = 0; i < n; i++) {
    parent[i] = i;
}
```

### Initial Representation

```text
0     1     2     3     4
|     |     |     |     |
0     1     2     3     4
```

Each node represents a separate set.

---

## 3. DSU Using Brute Force

### Approach

- Find the ultimate parent of both elements.
- If both have the same parent, they already belong to the same set.
- Otherwise, combine the two sets.
- Make one representative the parent of the other.

### Code

```cpp
// Find the ultimate parent of a node
int find(int i, vector<int>& parent) {

    if(i == parent[i])
        return i;

    return find(parent[i], parent);
}

// Combine two sets
void Union(int x, int y, vector<int>& parent) {

    int x_parent = find(x, parent);
    int y_parent = find(y, parent);

    // Already belong to the same set
    if(x_parent == y_parent)
        return;

    parent[x_parent] = y_parent;
}
```

### Complexity

| Operation | Time Complexity |
|---|---|
| Initialization | O(N) |
| Find | O(N) worst case |
| Union | O(N) worst case |

- Auxiliary recursive space: O(N) worst case.
- Parent array space: O(N).

**Problem:** Trees can become skewed, creating long chains. Finding the ultimate parent becomes expensive.

---

## 4. Optimization 1: Path Compression

### Approach

- Instead of simply returning the ultimate parent, update the parent of every visited node.
- Make every visited node directly point to the ultimate parent.
- This flattens the tree and makes future Find operations faster.

### Code

```cpp
int find(int i, vector<int>& parent) {

    if(i == parent[i])
        return i;

    return parent[i] = find(parent[i], parent);
}
```

### Important Statement

```cpp
parent[i] = find(parent[i], parent);
```

This statement:
1. Recursively finds the ultimate parent.
2. Updates the current node's parent.
3. Directly connects the current node to the ultimate parent.

### Dry Run

Before Path Compression:

```text
0 → 1 → 2 → 3
```

Call:

```cpp
find(0, parent);
```

Ultimate parent = `3`.

After Path Compression:

```text
0 ──→ 3
1 ──→ 3
2 ──→ 3
3 ──→ 3
```

### Complexity

- Single Find operation: O(N) worst case.
- Repeated operations: O(α(N)) amortized when combined with Union by Rank.

---

## 5. Optimization 2: Union by Rank

### Approach

- Rank represents the estimated height of a tree.
- Initially, the rank of every index is `0`.
- Find the ultimate parent of both nodes.
- Compare the ranks of their ultimate parents.
- The higher-rank element becomes the parent.
- If both ranks are equal, either can become the parent, and its rank increases by 1.

### Three Cases

**Case 1: Both ranks are equal**

- Make one parent the parent of the other.
- Increment the rank of the new parent.

```cpp
parent[x_parent] = y_parent;
rank[y_parent]++;
```

**Case 2: X has a greater rank**

- X's parent becomes the parent of Y's parent.
- Rank remains unchanged.

```cpp
parent[y_parent] = x_parent;
```

**Case 3: Y has a greater rank**

- Y's parent becomes the parent of X's parent.
- Rank remains unchanged.

```cpp
parent[x_parent] = y_parent;
```

### Complete Code

```cpp
// Union by Rank

void Union(int x, int y, vector<int>& parent,
           vector<int>& rank) {

    int x_parent = find(x, parent);
    int y_parent = find(y, parent);

    // Already belong to the same set
    if(x_parent == y_parent)
        return;

    // X has greater rank
    if(rank[x_parent] > rank[y_parent]) {

        parent[y_parent] = x_parent;
    }

    // Y has greater rank
    else if(rank[x_parent] < rank[y_parent]) {

        parent[x_parent] = y_parent;
    }

    // Both ranks are equal
    else {

        parent[x_parent] = y_parent;
        rank[y_parent]++;
    }
}
```

### Initialization

```cpp
int n = 5;

vector<int> parent(n);
vector<int> rank(n, 0);

for(int i = 0; i < n; i++) {
    parent[i] = i;
}
```

### Important Points

1. Always find the ultimate parent before comparing ranks.
2. Attach the smaller-rank tree below the larger-rank tree.
3. Increment rank only when both ranks are equal.
4

### Full Code
```cpp
#include <bits/stdc++.h>
using namespace std;

// Find with Path Compression
int find(int i, vector<int>& parent) {

    if (i == parent[i])
        return i;

    return parent[i] = find(parent[i], parent);
}

// Union by Rank
void Union(int x, int y, vector<int>& parent,
           vector<int>& rank) {

    int x_parent = find(x, parent);
    int y_parent = find(y, parent);

    // Already belong to the same set
    if (x_parent == y_parent)
        return;

    // X has greater rank
    if (rank[x_parent] > rank[y_parent]) {
        parent[y_parent] = x_parent;
    }

    // Y has greater rank
    else if (rank[x_parent] < rank[y_parent]) {
        parent[x_parent] = y_parent;
    }

    // Both ranks are equal
    else {
        parent[x_parent] = y_parent;
        rank[y_parent]++;
    }
}

int main() {

    int n = 5;

    // Initialization
    vector<int> parent(n);
    vector<int> rank(n, 0);

    // Initially every node is its own parent
    for (int i = 0; i < n; i++) {
        parent[i] = i;
    }

    // Perform Union operations
    Union(0, 1, parent, rank);
    Union(2, 3, parent, rank);
    Union(1, 2, parent, rank);

    // Check whether two elements belong to the same set
    int a = 0, b = 3;

    if (find(a, parent) == find(b, parent)) {
        cout << a << " and " << b << " belong to the same set.\n";
    }
    else {
        cout << a << " and " << b << " belong to different sets.\n";
    }

    // Display parent array
    cout << "\nParent Array: ";
    for (int i = 0; i < n; i++) {
        cout << parent[i] << " ";
    }

    // Display rank array
    cout << "\nRank Array: ";
    for (int i = 0; i < n; i++) {
        cout << rank[i] << " ";
    }

    return 0;
}
